<a href="https://colab.research.google.com/github/EoniseLeannePMagno/Final-Project-CS2/blob/main/rosal_students_DB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import requests
import json
url = "https://raw.githubusercontent.com/EoniseLeannePMagno/Final-Project-CS2/refs/heads/main/data/students.json"
response = requests.get(url)
subjects = response.json()
print(subjects)

[{'id': 1, 'name': 'Maria Santos', 'section': '7-Ampere', 'subjects': {'Math2': 1.5, 'Math3': 1.75, 'Biology1': 2.0, 'Chem1': 1.25, 'Physics1': 1.75, 'Computer Science2': 1.25, 'PEHM2': 1.0, 'VE2': 1.75, 'Social Science2': 1.5, 'English2': 1.25, 'Filipino2': 1.75}}, {'id': 2, 'name': 'Juan Dela Cruz', 'section': '7-Curie', 'subjects': {'Math2': 2.5, 'Math3': 2.75, 'Biology1': 2.0, 'Chem1': 2.25, 'Physics1': 2.5, 'Computer Science2': 2.0, 'PEHM2': 1.75, 'VE2': 2.5, 'Social Science2': 2.25, 'English2': 2.0, 'Filipino2': 2.75}}, {'id': 3, 'name': 'Ana Reyes', 'section': '7-Rutherford', 'subjects': {'Math2': 1.0, 'Math3': 1.25, 'Biology1': 1.5, 'Chem1': 1.0, 'Physics1': 1.25, 'Computer Science2': 1.0, 'PEHM2': 1.25, 'VE2': 1.25, 'Social Science2': 1.0, 'English2': 1.25, 'Filipino2': 1.5}}, {'id': 4, 'name': 'Carlo Mendoza', 'section': '7-Newton', 'subjects': {'Math2': 2.75, 'Math3': 3.0, 'Biology1': 2.5, 'Chem1': 2.75, 'Physics1': 2.5, 'Computer Science2': 2.25, 'PEHM2': 2.0, 'VE2': 2.5, '

In [3]:
!pip install firebase-admin


In [5]:
import firebase_admin
from firebase_admin import credentials, db
# Load the private key
cred = credentials.Certificate("/content/firebase_key (2).json")
# Initialize the app with your database URL
firebase_admin.initialize_app(cred, {
"databaseURL": "https://rosal-students-db-default-rtdb.asia-southeast1.firebasedatabase.app/"
})
print("Firebase connected successfully!")

Firebase connected successfully!


In [6]:
import requests
import json
"""
The 'subjects' variable is already loaded from the URL in a previous cell.
Therefore, we can directly use the 'subjects' variable.
"""
data = subjects
print("JSON file loaded")
ref = db.reference("students")
for student in data:
  ref.child(str(student["id"])).set(student)
print("Data uploaded successfully!")

JSON file loaded
Data uploaded successfully!


In [7]:
import firebase_admin
from firebase_admin import credentials, db

# Check if Firebase app is already initialized
try:
    firebase_admin.get_app()
except ValueError:
    # Firebase app not initialized, so initialize it
    cred = credentials.Certificate("/content/firebase_key.json")
    firebase_admin.initialize_app(cred, {
     "databaseURL": "https://rosal-students-db-default-rtdb.asia-southeast1.firebasedatabase.app/"
    })
    print("Firebase connected successfully (initialized in this cell)!")
else:
    print("Firebase app already initialized.")

student_name_to_search = input("Enter the name of the student to search: ")

ref = db.reference("students")
all_students = ref.get()

found = False

if all_students:
    for student_data in all_students:
        if student_data and "name" in student_data and student_name_to_search.lower() in student_data["name"].lower():
            print(f"\nRecord found for {student_name_to_search}:")
            print(f"  ID: {student_data['id']}")
            print(f"  Name: {student_data['name']}")
            print(f"  Section: {student_data['section']}")
            print("  Subjects:")
            if "subjects" in student_data:
                for subject, grade in student_data["subjects"].items():
                    print(f"    {subject}: {grade}")
            else:
                print("    No subjects recorded.")
            found = True
            break

if not found:
    print(f"\nRecord for '{student_name_to_search}' not found.")

Firebase app already initialized.
Enter the name of the student to search: Maria Santos

Record found for Maria Santos:
  ID: 1
  Name: Maria Santos
  Section: 7-Ampere
  Subjects:
    Biology1: 2.0
    Chem1: 1.25
    Computer Science2: 1.25
    English2: 1.25
    Filipino2: 1.75
    Math2: 1.5
    Math3: 1.75
    PEHM2: 1.0
    Physics1: 1.75
    Social Science2: 1.5
    VE2: 1.75


In [12]:
ref = db.reference("students")

while True:
    print("\n===== STUDENT DATABASE MENU ====")
    print("1. Display Students")
    print("2. Add Student")
    print("3. Update Student")
    print("4. Delete Student")
    print("5. Features")
    print("6. Exit")

    choice = input("Enter choice: ")

    # --- DISPLAY ---
    if choice == "1":
        students_data = ref.get()
        print("\nStudent List:")
        if students_data:
            # Firebase returns a list if IDs are sequential integers,
            # otherwise it returns a dictionary. We handle both here.
            items = students_data if isinstance(students_data, list) else students_data.values()

            for student_item in items:
                if student_item:
                    sid = student_item.get('id', 'N/A')
                    name = student_item.get('name', 'N/A')
                    print(f"ID: {sid}, Name: {name}")

                    if 'subjects' in student_item:
                        for sub, grade in student_item['subjects'].items():
                            print(f"    {sub}: {grade}")
        else:
            print("No students found.")

    # --- ADD ---
    elif choice == "2":
        while True:
            sid = input("Enter ID: ")
            if not sid.isdigit():
                print("Error: Student ID must be a number. Please enter a valid ID.")
            else:
                break # Valid ID, exit loop

        while True:
            name = input("Enter name: ")
            # Validate if the name is a number or can be converted to a float
            if name.strip().isdigit() or (len(name.strip()) > 0 and name.strip().replace('.', '', 1).isdigit()):
                print("Error: Student name cannot be a number. Please enter a valid name.")
            else:
                break # Valid name, exit loop

        section = input("Enter section: ")
        math = float(input("Enter Math2 grade: "))
        cs = float(input("Enter Computer Science2 grade: "))

        student = {
            "id": int(sid),
            "name": name,
            "section": section,
            "subjects": { "Math2": math, "Computer Science2": cs }
        }
        ref.child(sid).set(student)
        print("Student added successfully!")

    # --- UPDATE ---
    elif choice == "3":
        sid = input("Enter ID to update: ")
        student_ref = ref.child(sid)
        student = student_ref.get()

        if student:
            name = input(f"New name ({student.get('name')}): ")
            section = input(f"New section ({student.get('section')}): ")
            math = float(input("New Math2 grade: "))
            cs = float(input("New CS2 grade: "))

            student_ref.update(
                {
                    "name": name,
                    "section": section,
                    "subjects": { "Math2": math, "Computer Science2": cs }
                }
            )
            print("Update successful!")
        else:
            print("Student not found.")

    # --- DELETE ---
    elif choice == "4":
        sid = input("Enter ID to delete: ")
        if ref.child(sid).get():
            ref.child(sid).delete()
            print("Deleted successfully!")
        else:
            print("Student not found.")

    # --- FEATURES ---
    elif choice == "5":
        while True:
            print("\n---- FEATURES MENU ----")
            print("1. Compute General Average")
            print("2. List Subjects with Passing Grades (<= 2.50)")
            print("3. Compute Final Grade for a Subject")
            print("4. Organize Student Grades (Best to Worst)")
            print("5. List Subjects with Failing Grades (> 2.50)")
            print("6. Back to Main Menu")

            f_choice = input("Enter choice: ")

            if f_choice == "6":
                break

            sid = input("Enter Student ID: ")
            student = ref.child(sid).get()

            if not student or 'subjects' not in student:
                print("Error: Student not found or has no grades.")
                continue

            # Feature 1: Average
            if f_choice == "1":
                grades = student['subjects'].values()
                avg = sum(grades) / len(grades)
                print(f"Average for {student['name']}: {avg:.2f}")

            # Feature 2: Passing (Must be 2.50 or better/lower)
            elif f_choice == "2":
                print(f"Passing Subjects for {student['name']} (<= 2.50):")
                found = False
                for sub, grade in student['subjects'].items():
                    if grade <= 2.50:
                        print(f"  {sub}: {grade}")
                        found = True
                if not found:
                    print("  No passing subjects found.")

            # Feature 3: Specific Subject Grade
            elif f_choice == "3":
                sub_name = input("Enter exact subject name: ")
                grade = student['subjects'].get(sub_name)
                if grade is not None:
                    print(f"Grade for {sub_name}: {grade}")
                else:
                    print("Subject not found.")

            # Feature 4: Organize Grades (1.0 is the best, so we sort ascending)
            elif f_choice == "4":
                sorted_subs = sorted(student['subjects'].items(), key=lambda x: x[1])
                print(f"Grades for {student['name']} (Best to Worst):")
                for sub, grade in sorted_subs:
                    print(f"  {grade}: {sub}")

            # Feature 5: Failing (Anything worse/higher than 2.50)
            elif f_choice == "5":
                print(f"Failing Subjects for {student['name']} (> 2.50):")
                found = False
                for sub, grade in student['subjects'].items():
                    if grade > 2.50:
                        print(f"  {sub}: {grade}")
                        found = True
                if not found:
                    print("  No failing subjects found.")
            else:
                print("Invalid choice.")

    #EXIT
    elif choice == "6":
        print("Exiting. Goodbye!")
        break

    else:
        print("Invalid choice. Try again.")


===== STUDENT DATABASE MENU ====
1. Display Students
2. Add Student
3. Update Student
4. Delete Student
5. Features
6. Exit
Enter choice: 2
Enter ID: lol
Error: Student ID must be a number. Please enter a valid ID.
Enter ID: 89
Enter name: 89
Error: Student name cannot be a number. Please enter a valid name.
Enter name: lol
Enter section: io
Enter Math2 grade: 7


KeyboardInterrupt: Interrupted by user

In [13]:
all_subjects = set()
for student_data in subjects:
    if 'subjects' in student_data:
        all_subjects.update(student_data['subjects'].keys())

all_subjects = sorted(list(all_subjects))
print("All unique subjects identified:", all_subjects)

All unique subjects identified: ['Biology1', 'Chem1', 'Computer Science2', 'English2', 'Filipino2', 'Math2', 'Math3', 'PEHM2', 'Physics1', 'Social Science2', 'VE2']


In [ ]:
ref = db.reference("students")

try:
    while True:
        print("\n===== STUDENT DATABASE MENU ====")
        print("1. Display Students")
        print("2. Add Student")
        print("3. Update Student")
        print("4. Delete Student")
        print("5. Features")
        print("6. Exit")

        choice = input("Enter choice: ")

        # --- DISPLAY ---
        if choice == "1":
            students_data = ref.get()
            print("\nStudent List:")
            if students_data:
                # Firebase returns a list if IDs are sequential integers,
                # otherwise it returns a dictionary. We handle both here.
                items = students_data if isinstance(students_data, list) else students_data.values()

                for student_item in items:
                    if student_item:
                        sid = student_item.get('id', 'N/A')
                        name = student_item.get('name', 'N/A')
                        print(f"ID: {sid}, Name: {name}")

                        if 'subjects' in student_item:
                            for sub, grade in student_item['subjects'].items():
                                print(f"    {sub}: {grade}")
            else:
                print("No students found.")

        # --- ADD ---
        elif choice == "2":
            while True:
                sid = input("Enter ID: ")
                if not sid.isdigit():
                    print("Error: Student ID must be a number. Please enter a valid ID.")
                else:
                    break # Valid ID, exit loop

            while True:
                name = input("Enter name: ")
                # Validate if the name is a number or can be converted to a float
                if name.strip().isdigit() or (len(name.strip()) > 0 and name.strip().replace('.', '', 1).isdigit()):
                    print("Error: Student name cannot be a number. Please enter a valid name.")
                else:
                    break # Valid name, exit loop

            section = input("Enter section: ")

            new_student_subjects = {}
            for subject_name in all_subjects:
                while True:
                    try:
                        grade = float(input(f"Enter {subject_name} grade (1.00-5.00): "))
                        if 1.00 <= grade <= 5.00:
                            new_student_subjects[subject_name] = grade
                            break
                        else:
                            print("Error: Grade must be between 1.00 and 5.00.")
                    except ValueError:
                        print("Error: Invalid grade. Please enter a number.")

            student = {
                "id": int(sid),
                "name": name,
                "section": section,
                "subjects": new_student_subjects
            }
            ref.child(sid).set(student)
            print("Student added successfully!")

        # --- UPDATE ---
        elif choice == "3":
            sid = input("Enter ID to update: ")
            student_ref = ref.child(sid)
            student = student_ref.get()

            if student:
                name = input(f"New name ({student.get('name')}): ")
                section = input(f"New section ({student.get('section')}): ")

                updated_subjects = {}
                # Copy existing subjects or initialize if none
                if 'subjects' in student and student['subjects']:
                    updated_subjects.update(student['subjects'])

                # Loop through all known subjects to update or add new grades
                for subject_name in all_subjects:
                    current_grade = updated_subjects.get(subject_name, 'N/A')
                    while True:
                        grade_input = input(f"New {subject_name} grade ({current_grade}, enter to skip): ")
                        if not grade_input:
                            break # Skip if no input
                        try:
                            grade = float(grade_input)
                            if 1.00 <= grade <= 5.00:
                                updated_subjects[subject_name] = grade
                                break
                            else:
                                print("Error: Grade must be between 1.00 and 5.00.")
                        except ValueError:
                            print("Error: Invalid grade. Please enter a number.")

                student_ref.update(
                    {
                        "name": name,
                        "section": section,
                        "subjects": updated_subjects
                    }
                )
                print("Update successful!")
            else:
                print("Student not found.")

        # --- DELETE ---
        elif choice == "4":
            sid = input("Enter ID to delete: ")
            if ref.child(sid).get():
                ref.child(sid).delete()
                print("Deleted successfully!")
            else:
                print("Student not found.")

        # --- FEATURES ---
        elif choice == "5":
            while True:
                print("\n---- FEATURES MENU ----")
                print("1. Compute General Average")
                print("2. List Subjects with Passing Grades (<= 2.50)")
                print("3. Compute Final Grade for a Subject")
                print("4. Organize Student Grades (Best to Worst)")
                print("5. List Subjects with Failing Grades (> 2.50)")
                print("6. Back to Main Menu")

                f_choice = input("Enter choice: ")

                if f_choice == "6":
                    break

                sid = input("Enter Student ID: ")
                student = ref.child(sid).get()

                if not student or 'subjects' not in student:
                    print("Error: Student not found or has no grades.")
                    continue

                # Feature 1: Average
                if f_choice == "1":
                    grades = student['subjects'].values()
                    avg = sum(grades) / len(grades)
                    print(f"Average for {student['name']}: {avg:.2f}")

                # Feature 2: Passing (Must be 2.50 or better/lower)
                elif f_choice == "2":
                    print(f"Passing Subjects for {student['name']} (<= 2.50):")
                    found = False
                    for sub, grade in student['subjects'].items():
                        if grade <= 2.50:
                            print(f"  {sub}: {grade}")
                            found = True
                    if not found:
                        print("  No passing subjects found.")

                # Feature 3: Specific Subject Grade
                elif f_choice == "3":
                    sub_name = input("Enter exact subject name: ")
                    grade = student['subjects'].get(sub_name)
                    if grade is not None:
                        print(f"Grade for {sub_name}: {grade}")
                    else:
                        print("Subject not found.")

                # Feature 4: Organize Grades (1.0 is the best, so we sort ascending)
                elif f_choice == "4":
                    sorted_subs = sorted(student['subjects'].items(), key=lambda x: x[1])
                    print(f"Grades for {student['name']} (Best to Worst):")
                    for sub, grade in sorted_subs:
                        print(f"  {grade}: {sub}")

                # Feature 5: Failing (Anything worse/higher than 2.50)
                elif f_choice == "5":
                    print(f"Failing Subjects for {student['name']} (> 2.50):")
                    found = False
                    for sub, grade in student['subjects'].items():
                        if grade > 2.50:
                            print(f"  {sub}: {grade}")
                            found = True
                    if not found:
                        print("  No failing subjects found.")
                else:
                    print("Invalid choice.")

        #EXIT
        elif choice == "6":
            print("Exiting. Goodbye!")
            break

        else:
            print("Invalid choice. Try again.")
except KeyboardInterrupt:
    print("\nMenu interrupted by user. Exiting gracefully.")
